In [ ]:
import pandas as pd
import os

# Identificar onde estamos
print(f"Diretório atual de trabalho: {os.getcwd()}")

# Definir o caminho relativo para os dados
path_dados = '../data/raw'

# Listar o que tem no cofre
try:
    arquivos = os.listdir(path_dados)
    print(f"\nArquivos encontrados em '{path_dados}':")
    for f in arquivos:
        print(f" - {f}")
except FileNotFoundError:
    print(f"\nERRO CRÍTICO: Não encontrei a pasta {path_dados}.")

# Teste de Carga: Ler o arquivo de Clientes
arquivo_clientes = os.path.join(path_dados, 'olist_customers_dataset.csv')

if os.path.exists(arquivo_clientes):
    print("\n--- Carregando Dataset de Clientes ---")
    df_clientes = pd.read_csv(arquivo_clientes)
    print("Sucesso! Amostra dos dados:")
    display(df_clientes.head()) # Mostra as 5 primeiras linhas formatadas
else:
    print(f"\nERRO: O arquivo {arquivo_clientes} não existe!")

print("\n")
# Verificar as dimensões de arquivo ( linhas e colunas )
print(f"A quantidade de linhas e colunas é de: {df_clientes.shape}\n")

# Verificar os tipos de dados
print("Tipos de dados encontrados:\n")
df_clientes.info()

# Resumo estatístico
df_clientes.describe()


print("\n")
# Realizar a contagem absoluta de quantos clientes tenho por estado
print("### Clientes por Estado (Absoluto) ###")
display(df_clientes['customer_state'].value_counts().head())
print("\n")

# Verificar a porcentagem de clientes por estado
print("### Clientes por estado (%) ###")

# Calcular a porcentagem correta e transformar em numeros legiveis
porcentagem_estados = df_clientes['customer_state'].value_counts(normalize=True)
porcentagem_legivel = porcentagem_estados * 100

# Exibir com porcentagem correta
display(porcentagem_legivel.head())

In [ ]:
# Carregamento do dataset de Pedidos

# Identificar onde estamos
print(f"Diretório atual de trabalho: {os.getcwd()}")

# Definir o caminho relativo dos dados
path_dados = '../data/raw'

try:
    arquivos = os.listdir(path_dados)
    print(f"Arquivos encontrados em '{path_dados}': ")
    for f in arquivos:
        print(f"- {f}")
except FileNotFoundError:
    print(f"ERRO CRÍTICO: Não encontrei a pasta {path_dados}.")

# Teste de carga: Ler arquivo de pedidos
arquivo_pedidos = os.path.join(path_dados, "olist_orders_dataset.csv")

if os.path.exists(arquivo_pedidos):
    print("\n--- Carregando Dataset de Pedidos ---")
    df_pedidos = pd.read_csv(arquivo_pedidos)
    print("Sucesso! Amostra dos dados:")
    display(df_pedidos.head()) # Para primeiras 5 linhas formatadas
else:
    print(f"\nERRO: O arquivo {arquivo_pedidos} não existe!")

# Verificar os tipos de dados
print(f"Tipos de dados encontrados:\n")
df_pedidos.info()
print("\n")


# Conversão de datas em object para datetime
cols_conversao = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

print(f"### Realizando a conversão das colunas ###\n")

for col in cols_conversao:
    print(f"Convertendo a coluna: {col}...")
    df_pedidos[col] = pd.to_datetime(df_pedidos[col])

# Verificação se Dtype foi alterado
df_pedidos.info()
print("\n")


#  Join/Merge para enriquecimento de dados clientes x pedidos
print(f"Realizando merge de informações...\n")
df_completo = df_pedidos.merge(
    df_clientes,
    on='customer_id',
    how='inner'
)

# Nova verificação sobre merge
df_completo.info()

In [ ]:
# Verificação de timedelta da entrega/compra de pedidos

# Filtragem
df_final = df_completo[df_completo['order_status'] == 'delivered'].copy()

# Verificação de quantas linhas sobraram
print(f"Total original: {df_completo.shape[0]}")
print(f"Total entregue: {df_final.shape[0]}")


# Extrair o número de dias de diferença
df_final['tempo_entrega_dias'] = (df_final['order_delivered_customer_date'] - df_final['order_purchase_timestamp']).dt.days

# Validação
cols_para_confirmar = ['order_purchase_timestamp',
                       'order_delivered_customer_date',
                       'tempo_entrega_dias']

display(df_final[cols_para_confirmar].head())

In [ ]:
# Agregação para saber entregas por estados (GroupBy)
media_por_estado = df_final.groupby('customer_state')['tempo_entrega_dias'].mean()

# Validar do pior para o melhor
media_por_estado = media_por_estado.sort_values(ascending=False)

# Visualizar resultado
print("### Média de dias de entrega por Estado ( Pior para melhor ) ###\n")
display(media_por_estado.head(10))

In [42]:
# Etapa de Load das informações

# Definir local para salvar arquivo
caminho_final = '../data/processed/olist_processed.csv'

# Salvar o arquivo
df_final.to_csv(caminho_final, index=False)

print(f"Arquivo salvo com sucesso em: {caminho_final}")


# Confirmação de arquivo salvo
import os

if os.path.exists(caminho_final):
    print("Arquivo salvo e pronto para utilização com MySQL.")
else:
    print("Erro: Arquivo não encontrado.")

Arquivo salvo com sucesso em: ../data/processed/olist_processed.csv
Arquivo salvo e pronto para utilização com MySQL.
